# Two-File Audio Fault Analyzer

FFT and STFT analysis of two recordings. Original exploratory project: **23 March 2026 (author-reported)**. Restored code and evidence export: **10 October 2026**.

The supplied example compares an engine recording with a drum loop. It is a signal-analysis demonstration, not a validated engine diagnosis. The original hand-built labels and severity are retained for reproduction, including both file orders. The Welch band-power plot is separately labelled October analysis.

Download the recordings from their original Pixabay pages: [engine](https://pixabay.com/sound-effects/city-diesel-car-engine-sound-111994/) and [drum](https://pixabay.com/sound-effects/timbo-drumline-loop-103bpm-171091/). See the repository README and audit for measurement definitions.


In [ ]:
# Install the standard dependencies and restore the embedded source.
import subprocess, sys
from pathlib import Path
subprocess.run(['apt-get','-y','install','ffmpeg'],check=True,stdout=subprocess.DEVNULL)
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','scipy','matplotlib','pydub==0.25.1'],check=True)
work=Path('/content/audio_fft_project')
work.mkdir(parents=True,exist_ok=True)
(work / 'original_prototype.py').write_text('"""Analysis functions from the author\'s supplied March 2026 Colab prototype.\n\nThe feature definitions and heuristics are preserved for reproduction.\nThe upload cell and plotting/printing runner are provided separately.\nThese heuristic scores are not validated engine-fault diagnoses.\n"""\nimport numpy as np\nfrom pydub import AudioSegment\nfrom scipy.signal import find_peaks, get_window, stft, butter, filtfilt, savgol_filter\n\nEPS = 1e-12\n\ndef next_pow2(n):\n    return 1 if n <= 1 else 2 ** int(np.ceil(np.log2(n)))\n\ndef clamp01(x):\n    return float(max(0.0, min(1.0, x)))\n\ndef safe_mean(x):\n    x = np.asarray(x, dtype=float)\n    return float(np.nanmean(x)) if len(x) else np.nan\n\ndef safe_std(x):\n    x = np.asarray(x, dtype=float)\n    return float(np.nanstd(x)) if len(x) else np.nan\n\ndef robust_zscore(x):\n    x = np.asarray(x, dtype=float)\n    med = np.nanmedian(x)\n    mad = np.nanmedian(np.abs(x - med)) + EPS\n    return 0.6745 * (x - med) / mad\n\ndef smooth_series(x, win=11, poly=2):\n    x = np.asarray(x, dtype=float)\n    if len(x) < 7:\n        return x\n    win = min(win, len(x) if len(x) % 2 == 1 else len(x) - 1)\n    if win < 5:\n        return x\n    if win % 2 == 0:\n        win += 1\n    poly = min(poly, win - 2)\n    return savgol_filter(x, window_length=win, polyorder=poly)\n\ndef load_audio(path):\n    audio = AudioSegment.from_file(path)\n    raw = np.array(audio.get_array_of_samples())\n    if audio.channels > 1:\n        raw = raw.reshape((-1, audio.channels)).mean(axis=1)\n    scale = float(2 ** (8 * audio.sample_width - 1))\n    x = raw.astype(np.float32) / scale\n    x = np.nan_to_num(x)\n    x = np.clip(x, -1.0, 1.0)\n    return x, audio.frame_rate, audio.channels, audio.sample_width\n\ndef highpass(x, fs, cutoff=20.0, order=3):\n    nyq = 0.5 * fs\n    cutoff = min(cutoff, nyq * 0.95)\n    b, a = butter(order, cutoff / nyq, btype="highpass")\n    return filtfilt(b, a, x)\n\ndef compute_spectrum(x, fs, n_fft=None):\n    x = np.asarray(x, dtype=np.float32)\n    x = x - np.mean(x)\n    if n_fft is None:\n        n_fft = next_pow2(len(x))\n    n_fft = max(n_fft, 512)\n    window = get_window("hann", len(x), fftbins=True)\n    xw = x * window\n    spec = np.fft.rfft(xw, n=n_fft)\n    mag = np.abs(spec)\n    freqs = np.fft.rfftfreq(n_fft, d=1.0 / fs)\n    return freqs, mag\n\ndef band_energy(freqs, mag, lo, hi):\n    mask = (freqs >= lo) & (freqs < hi)\n    return float(np.sum(mag[mask]) / (np.sum(mag) + EPS))\n\ndef harmonic_search(peak_freqs, fs):\n    best_f0 = np.nan\n    best_score = 0.0\n    matched = []\n    if len(peak_freqs) == 0:\n        return best_f0, best_score, matched\n    candidate_pool = peak_freqs[(peak_freqs >= 20) & (peak_freqs <= min(2000, fs / 4))]\n    candidate_pool = candidate_pool[:20] if len(candidate_pool) > 20 else candidate_pool\n    for f0 in candidate_pool:\n        if f0 <= 0:\n            continue\n        max_h = int((fs / 2) // f0)\n        max_h = max(2, min(max_h, 12))\n        hits = []\n        checks = 0\n        for k in range(2, max_h + 1):\n            expected = k * f0\n            tol = max(4.0, 0.03 * expected)\n            checks += 1\n            close = peak_freqs[np.abs(peak_freqs - expected) <= tol]\n            if len(close) > 0:\n                hits.append(float(close[0]))\n        score = len(hits) / max(1, checks)\n        if score > best_score:\n            best_score = score\n            best_f0 = float(f0)\n            matched = hits\n    return best_f0, float(best_score), matched\n\ndef extract_global_features(x, fs):\n    x = highpass(x, fs, cutoff=20.0, order=3)\n    peak_amp = float(np.max(np.abs(x)))\n    rms = float(np.sqrt(np.mean(x ** 2)))\n    dc_offset = float(np.mean(x))\n    crest_factor = float(peak_amp / (rms + EPS))\n    clipping_ratio = float(np.mean(np.abs(x) >= 0.98))\n    zcr = float(np.mean(np.abs(np.diff(np.signbit(x)))))\n    freqs, mag = compute_spectrum(x, fs, n_fft=next_pow2(len(x)))\n    p = mag / (np.sum(mag) + EPS)\n    dom_i = int(np.argmax(mag))\n    dominant_freq = float(freqs[dom_i])\n    dominant_amp = float(mag[dom_i])\n    centroid = float(np.sum(freqs * p))\n    spread = float(np.sqrt(np.sum(((freqs - centroid) ** 2) * p)))\n    flatness = float(np.exp(np.mean(np.log(mag + EPS))) / (np.mean(mag) + EPS))\n    entropy = float(-np.sum(p * np.log2(p + EPS)))\n    crest_spectrum = float(np.max(mag) / (np.mean(mag) + EPS))\n    cdf = np.cumsum(p)\n    def rolloff(frac):\n        idx = int(np.searchsorted(cdf, frac))\n        idx = min(idx, len(freqs) - 1)\n        return float(freqs[idx])\n    roll50 = rolloff(0.50)\n    roll85 = rolloff(0.85)\n    roll95 = rolloff(0.95)\n    valid = freqs > 0\n    if np.sum(valid) >= 8:\n        slope = float(np.polyfit(np.log10(freqs[valid]), np.log10(mag[valid] + EPS), 1)[0])\n    else:\n        slope = np.nan\n    noise_floor = float(np.median(mag))\n    prominence = max(np.max(mag) * 0.03, noise_floor * 3.0, EPS)\n    distance_bins = max(1, int(20 / max(freqs[1] - freqs[0], 1e-9))) if len(freqs) > 1 else 1\n    peaks, _ = find_peaks(mag, prominence=prominence, distance=distance_bins)\n    peak_freqs = freqs[peaks] if len(peaks) else np.array([])\n    peak_amps = mag[peaks] if len(peaks) else np.array([])\n    if len(peaks) > 0:\n        order = np.argsort(peak_amps)[::-1]\n        peak_freqs = peak_freqs[order]\n        peak_amps = peak_amps[order]\n    best_f0, harmonic_score, matched_harmonics = harmonic_search(peak_freqs, fs)\n    bands = [(0, 60), (60, 250), (250, 500), (500, 1000), (1000, 3000), (3000, 8000), (8000, fs / 2)]\n    band_shares = [band_energy(freqs, mag, lo, hi) for lo, hi in bands]\n    return {\n        "x": x, "fs": fs, "duration": len(x) / fs, "samples": len(x),\n        "peak_amp": peak_amp, "rms": rms, "dc_offset": dc_offset,\n        "crest_factor": crest_factor, "clipping_ratio": clipping_ratio, "zcr": zcr,\n        "freqs": freqs, "mag": mag, "dominant_freq": dominant_freq, "dominant_amp": dominant_amp,\n        "centroid": centroid, "spread": spread, "flatness": flatness, "entropy": entropy,\n        "crest_spectrum": crest_spectrum, "roll50": roll50, "roll85": roll85, "roll95": roll95,\n        "slope": slope, "noise_floor": noise_floor, "peak_count": int(len(peaks)),\n        "peak_freqs": peak_freqs, "peak_amps": peak_amps, "best_f0": best_f0,\n        "harmonic_score": harmonic_score, "matched_harmonics": matched_harmonics,\n        "band_shares": band_shares,\n    }\n\ndef frame_features(x, fs, frame_sec=0.03, hop_sec=0.01):\n    frame_len = max(256, int(frame_sec * fs))\n    hop_len = max(64, int(hop_sec * fs))\n    names = ["time", "rms", "zcr", "dominant_freq", "centroid", "spread", "flatness", "entropy", "crest", "spectral_flux", "peak_count", "harmonic_score"]\n    values = {name: [] for name in names}\n    prev_mag_norm = None\n    for start in range(0, len(x) - frame_len + 1, hop_len):\n        frame = x[start:start + frame_len]\n        frame = frame - np.mean(frame)\n        freqs, mag = compute_spectrum(frame, fs, n_fft=next_pow2(frame_len))\n        p = mag / (np.sum(mag) + EPS)\n        dom_i = int(np.argmax(mag))\n        dom = float(freqs[dom_i])\n        centroid = float(np.sum(freqs * p))\n        spread = float(np.sqrt(np.sum(((freqs - centroid) ** 2) * p)))\n        flatness = float(np.exp(np.mean(np.log(mag + EPS))) / (np.mean(mag) + EPS))\n        entropy = float(-np.sum(p * np.log2(p + EPS)))\n        crest = float(np.max(mag) / (np.mean(mag) + EPS))\n        noise_floor = float(np.median(mag))\n        prominence = max(np.max(mag) * 0.03, noise_floor * 3.0, EPS)\n        distance_bins = max(1, int(20 / max(freqs[1] - freqs[0], 1e-9))) if len(freqs) > 1 else 1\n        peaks, _ = find_peaks(mag, prominence=prominence, distance=distance_bins)\n        peak_freqs = freqs[peaks] if len(peaks) else np.array([])\n        peak_amps = mag[peaks] if len(peaks) else np.array([])\n        if len(peaks) > 0:\n            order = np.argsort(peak_amps)[::-1]\n            peak_freqs = peak_freqs[order]\n        _, harmonic_score, _ = harmonic_search(peak_freqs, fs)\n        mag_norm = mag / (np.sum(mag) + EPS)\n        if prev_mag_norm is None:\n            flux = 0.0\n        else:\n            flux = float(np.sum(np.maximum(mag_norm - prev_mag_norm, 0.0)))\n        prev_mag_norm = mag_norm\n        rms = float(np.sqrt(np.mean(frame ** 2)))\n        zcr = float(np.mean(np.abs(np.diff(np.signbit(frame)))))\n        row = [((start + frame_len / 2) / fs), rms, zcr, dom, centroid, spread, flatness, entropy, crest, flux, int(len(peaks)), harmonic_score]\n        for key, value in zip(names, row):\n            values[key].append(value)\n    return {key: np.array(value) for key, value in values.items()}\n\ndef analyze_file(path):\n    x, fs, channels, sample_width = load_audio(path)\n    x = highpass(x, fs, cutoff=20.0, order=3)\n    g = extract_global_features(x, fs)\n    fr = frame_features(g["x"], fs, frame_sec=0.03, hop_sec=0.01)\n    Z = np.vstack([robust_zscore(fr[key]) for key in ["rms", "zcr", "dominant_freq", "centroid", "flatness", "entropy", "crest", "spectral_flux", "peak_count", "harmonic_score"]]).T\n    anomaly_score = np.nanmax(np.abs(Z), axis=1)\n    fr["anomaly_score"] = anomaly_score\n    threshold = 4.0\n    anomaly_idx = np.where(anomaly_score > threshold)[0]\n    anomaly_rate = float(len(anomaly_idx) / max(1, len(anomaly_score)))\n    flux_sm = smooth_series(fr["spectral_flux"], win=9, poly=2) if len(fr["spectral_flux"]) >= 9 else fr["spectral_flux"]\n    flux_z = robust_zscore(flux_sm)\n    onset_idx, _ = find_peaks(flux_z, prominence=1.2, distance=max(2, int(0.08 / 0.01)))\n    onset_times = fr["time"][onset_idx]\n    onset_density = float(len(onset_times) / max(g["duration"], EPS))\n    def estimate_decay_time(x, fs, onset_time, search_sec=0.7, rel_level=0.25):\n        start = int(onset_time * fs)\n        end = min(len(x), start + int(search_sec * fs))\n        if start >= len(x) - 1 or end <= start + 10:\n            return np.nan, np.nan\n        seg = x[start:end]\n        env = np.abs(seg)\n        peak = float(np.max(env))\n        if peak <= EPS:\n            return np.nan, np.nan\n        thr = peak * rel_level\n        hit = np.where(env <= thr)[0]\n        decay = float(hit[0] / fs) if len(hit) else float((end - start) / fs)\n        return peak, decay\n    hit_peaks = []\n    hit_decays = []\n    for t0 in onset_times:\n        p, d = estimate_decay_time(g["x"], fs, t0)\n        hit_peaks.append(p)\n        hit_decays.append(d)\n    return {\n        "file": path, "fs": fs, "channels": channels, "sample_width": sample_width,\n        "global": g, "frames": fr, "anomaly_rate": anomaly_rate, "anomaly_threshold": threshold,\n        "anomaly_idx": anomaly_idx, "onset_idx": onset_idx, "onset_times": onset_times,\n        "onset_density": onset_density, "hit_peaks": np.array(hit_peaks, dtype=float),\n        "hit_decays": np.array(hit_decays, dtype=float), "flux_z": flux_z,\n    }\n\ndef score_file(result):\n    g = result["global"]\n    fr = result["frames"]\n    mean_flux = safe_mean(fr["spectral_flux"])\n    mean_harm = safe_mean(fr["harmonic_score"])\n    mean_flat = safe_mean(fr["flatness"])\n    mean_entropy = safe_mean(fr["entropy"])\n    mean_crest = safe_mean(fr["crest"])\n    dom_std = safe_std(fr["dominant_freq"])\n    anomaly_rate = result["anomaly_rate"]\n    onset_density = result["onset_density"]\n    high_band = g["band_shares"][5] + g["band_shares"][6]\n    low_band = g["band_shares"][0] + g["band_shares"][1]\n    high_low_ratio = float((high_band + EPS) / (low_band + EPS))\n    percussive_score = (\n        0.25 * clamp01(mean_flux / 0.05) + 0.20 * clamp01(onset_density / 8.0) +\n        0.15 * clamp01(mean_crest / 12.0) + 0.15 * clamp01(anomaly_rate / 0.25) +\n        0.10 * clamp01(mean_entropy / 12.0) + 0.15 * (1.0 - clamp01(mean_harm)))\n    continuous_score = (\n        0.35 * clamp01(mean_harm) + 0.15 * (1.0 - clamp01(mean_flux / 0.05)) +\n        0.10 * (1.0 - clamp01(onset_density / 8.0)) + 0.10 * (1.0 - clamp01(mean_flat / 0.5)))\n    noise_score = (\n        0.25 * clamp01(mean_flat / 0.3) + 0.20 * clamp01(mean_entropy / 12.0) +\n        0.20 * clamp01(high_low_ratio / 1.2) + 0.20 * clamp01(g["centroid"] / 4000.0) +\n        0.15 * clamp01(g["roll95"] / (g["fs"] / 2)))\n    instability_score = (\n        0.35 * clamp01(dom_std / 20.0) + 0.25 * clamp01(mean_flux / 0.05) +\n        0.20 * clamp01(anomaly_rate / 0.25) + 0.20 * clamp01(safe_std(fr["harmonic_score"]) / 0.35))\n    return {\n        "percussive_score": percussive_score, "continuous_score": continuous_score,\n        "noise_score": noise_score, "instability_score": instability_score,\n        "mean_flux": mean_flux, "mean_harm": mean_harm, "mean_flat": mean_flat,\n        "mean_entropy": mean_entropy, "mean_crest": mean_crest, "dom_std": dom_std,\n        "high_low_ratio": high_low_ratio, "anomaly_rate": anomaly_rate, "onset_density": onset_density,\n    }\n\ndef verdict_from_scores(scores):\n    p, c, n, u = (scores[key] for key in ["percussive_score", "continuous_score", "noise_score", "instability_score"])\n    if p > c + 0.12 and p > 0.45:\n        base = "Likely percussive / drum-like"\n    elif c > p + 0.12 and c > 0.40:\n        base = "Likely continuous / tonal / mechanical"\n    else:\n        base = "Mixed or ambiguous"\n    flags = []\n    if n > 0.55:\n        flags.append("noisy / rough")\n    if u > 0.55:\n        flags.append("unstable over time")\n    if not flags:\n        flags.append("fairly stable")\n    return base + " | " + ", ".join(flags)\n\ndef compare_results(base_res, test_res):\n    b, t = base_res["global"], test_res["global"]\n    bs, ts = score_file(base_res), score_file(test_res)\n    def rel_change(a, b):\n        return float((b - a) / (abs(a) + EPS))\n    shifts = {key: rel_change(b[key], t[key]) for key in ["dominant_freq", "centroid", "flatness", "entropy", "crest_factor", "harmonic_score", "peak_count", "roll95"]}\n    shifts.update({"anomaly_rate": rel_change(base_res["anomaly_rate"], test_res["anomaly_rate"]),\n                   "onset_density": rel_change(base_res["onset_density"], test_res["onset_density"]),\n                   "high_low_ratio": rel_change(bs["high_low_ratio"], ts["high_low_ratio"])})\n    hints = []\n    if t["harmonic_score"] < b["harmonic_score"] - 0.15 or t["harmonic_score"] < 0.35:\n        hints.append("possible irregular timing / missing structure")\n    if t["flatness"] > b["flatness"] + 0.08 or t["noise_floor"] > b["noise_floor"] * 1.25:\n        hints.append("possible extra noise / wear / roughness")\n    if ts["high_low_ratio"] > bs["high_low_ratio"] * 1.2 or t["centroid"] > b["centroid"] * 1.15:\n        hints.append("more high-frequency energy than baseline")\n    if test_res["onset_density"] > base_res["onset_density"] * 1.5:\n        hints.append("more transient impacts / knocks / drum-like events")\n    if test_res["anomaly_rate"] > base_res["anomaly_rate"] * 1.5:\n        hints.append("time-varying instability increased")\n    if abs(t["dominant_freq"] - b["dominant_freq"]) > max(15.0, 0.12 * b["dominant_freq"]):\n        hints.append("main frequency shifted noticeably")\n    if t["crest_factor"] > b["crest_factor"] * 1.2:\n        hints.append("more spiky / impulsive waveform")\n    if not hints:\n        hints.append("no strong fault signature compared to baseline")\n    severity = (\n        0.25 * clamp01(abs(shifts["harmonic_score"])) + 0.20 * clamp01(abs(shifts["flatness"])) +\n        0.15 * clamp01(abs(shifts["centroid"])) + 0.15 * clamp01(abs(shifts["anomaly_rate"])) +\n        0.10 * clamp01(abs(shifts["onset_density"])) + 0.15 * clamp01(abs(shifts["high_low_ratio"])))\n    return shifts, hints, severity\n')
(work / 'run_analysis.py').write_text('"""Reproduce the supplied prototype and export explicitly labelled evidence.\n\npython run_analysis.py --engine /path/engine.mp3 --comparison /path/drum.mp3\nRequires ffmpeg on PATH. Audio sources are downloaded by the reader separately.\n"""\nfrom pathlib import Path\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport platform\nimport subprocess\nimport sys\nimport numpy as np\nimport scipy\nfrom scipy.signal import resample_poly, stft, welch\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom original_prototype import analyze_file, score_file, compare_results, verdict_from_scores, load_audio, highpass\n\nROOT = Path(__file__).resolve().parent\nNAVY, TEAL, ORANGE = \'#132b46\', \'#007f87\', \'#bd5f27\'\nplt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':16,\'axes.labelsize\':17,\n                     \'axes.titlesize\':19,\'axes.titleweight\':\'bold\',\'axes.spines.top\':False,\n                     \'axes.spines.right\':False,\'axes.edgecolor\':\'#becbd4\',\'text.color\':NAVY,\n                     \'axes.labelcolor\':NAVY,\'xtick.color\':NAVY,\'ytick.color\':NAVY,\n                     \'figure.facecolor\':\'white\',\'axes.facecolor\':\'white\'})\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef summary(result):\n    g=result[\'global\']\n    return {\'file\':Path(result[\'file\']).name,\'sample_rate_hz\':result[\'fs\'],\'channels\':result[\'channels\'],\n            \'duration_s\':g[\'duration\'],**{k:g[k] for k in [\'peak_amp\',\'rms\',\'crest_factor\',\'clipping_ratio\',\'dominant_freq\',\'centroid\',\'flatness\',\'entropy\',\'harmonic_score\',\'peak_count\',\'roll95\']},\n            \'magnitude_band_shares\':g[\'band_shares\'],\'anomaly_rate\':result[\'anomaly_rate\'],\n            \'onset_density\':result[\'onset_density\'],\'prototype_verdict\':verdict_from_scores(score_file(result)),\n            \'top_peak_frequencies_hz\':g[\'peak_freqs\'][:10].tolist(),\n            \'top_peak_magnitudes\':g[\'peak_amps\'][:10].tolist()}\n\ndef print_original(tag,result):\n    g=result[\'global\'];s=summary(result)\n    print(\'\\n\'+\'=\'*70+\'\\n\'+tag+\'\\n\'+\'=\'*70)\n    for label,key in [(\'File\',\'file\'),(\'Sample rate\',\'sample_rate_hz\'),(\'Channels\',\'channels\'),(\'Duration\',\'duration_s\'),(\'Peak amplitude\',\'peak_amp\'),(\'RMS\',\'rms\'),(\'Crest factor\',\'crest_factor\'),(\'Clipping ratio\',\'clipping_ratio\'),(\'Dominant freq\',\'dominant_freq\'),(\'Centroid\',\'centroid\'),(\'Flatness\',\'flatness\'),(\'Entropy\',\'entropy\'),(\'Harmonic score\',\'harmonic_score\'),(\'Peak count\',\'peak_count\'),(\'Anomaly rate\',\'anomaly_rate\'),(\'Onset density\',\'onset_density\'),(\'Prototype verdict\',\'prototype_verdict\')]:\n        print(f\'{label:20}: {s[key]}\')\n    print(\'Top peaks (Hz, unnormalised magnitude):\')\n    for f,a in zip(g[\'peak_freqs\'][:10],g[\'peak_amps\'][:10]):\n        print(f\'  {f:.2f} Hz | {a:.6f}\')\n    print(\'Magnitude band shares (original code labels these energy):\',g[\'band_shares\'])\n    print(\'First ten onset estimates (time, peak, original decay estimate):\')\n    for t,p,d in zip(result[\'onset_times\'][:10],result[\'hit_peaks\'][:10],result[\'hit_decays\'][:10]):\n        print(f\'  {t:.3f}s | {p:.6f} | {d:.6f}s\')\n\ndef make_figures(engine,comparison,out):\n    fig,ax=plt.subplots(figsize=(12,5.3),layout=\'constrained\')\n    for r,label,color in [(engine,\'Engine recording · reference\',TEAL),(comparison,\'Drum loop · comparison\',ORANGE)]:\n        g=r[\'global\'];w=np.hanning(len(g[\'x\'])+1)[:-1]\n        a=2*g[\'mag\']/w.sum()\n        ax.plot(g[\'freqs\'],a,label=label,color=color,lw=1.35,alpha=.9)\n        idx=int(np.argmax(g[\'mag\']));f=float(g[\'freqs\'][idx]);v=float(a[idx])\n        ax.annotate(f\'{f:.2f} Hz\',(f,v),xytext=(f+80,v*1.08),color=color,fontsize=17,\n                    arrowprops={\'arrowstyle\':\'-\',\'color\':color})\n    peak_height=max(2*r[\'global\'][\'mag\'].max()/np.hanning(len(r[\'global\'][\'x\'])+1)[:-1].sum() for r in [engine,comparison])\n    ax.set(xlim=(0,1600),ylim=(0,peak_height*1.25),xlabel=\'Frequency (Hz)\',ylabel=\'Window-normalised FFT amplitude\',title=\'Different sounds, different frequency structure\')\n    ax.grid(axis=\'both\',alpha=.18);ax.legend(frameon=False,fontsize=14,loc=\'upper right\')\n    fig.text(.5,-.028,\'Full recordings · Hann window · two-file demonstration; no fault labels\',ha=\'center\',fontsize=13,color=\'#4c6074\')\n    fig.savefig(out/\'fft_comparison.png\',dpi=170,bbox_inches=\'tight\');plt.close(fig)\n\n    fs=44100\n    ys=[]\n    for r in [engine,comparison]:\n        y,rate,_,_=load_audio(r[\'file\'])\n        y=highpass(y,rate,20.0,3)\n        if rate!=fs:\n            from math import gcd\n            d=gcd(rate,fs);y=resample_poly(y,fs//d,rate//d)\n        ys.append(y)\n    n=min(map(len,ys));duration=n/fs\n    ys=[y[:n] for y in ys]\n    fig,axes=plt.subplots(2,1,figsize=(12,5.5),sharex=True,sharey=True,layout=\'constrained\')\n    for ax,y,label in zip(axes,ys,[\'Engine reference\',\'Drum comparison\']):\n        y=y/(np.sqrt(np.mean(y*y))+1e-12)\n        f,t,z=stft(y,fs=fs,window=\'hann\',nperseg=2048,noverlap=1536,boundary=None,padded=False)\n        im=ax.pcolormesh(t,f/1000,20*np.log10(np.abs(z)+1e-12),shading=\'auto\',vmin=-70,vmax=0,cmap=\'magma\',rasterized=True)\n        ax.set_ylim(0,4);ax.set_ylabel(\'kHz\');ax.text(.015,.85,label,transform=ax.transAxes,color=\'white\',weight=\'bold\',fontsize=16)\n    axes[0].set_title(\'How frequency content changes through time\')\n    axes[-1].set_xlabel(\'Time (s)\');fig.colorbar(im,ax=axes,label=\'dB relative to unit RMS\',pad=.018)\n    fig.text(.5,-.025,f\'Same first {duration:.2f} s · 44.1 kHz · 2,048-sample Hann window · 512-sample hop\',ha=\'center\',fontsize=13,color=\'#4c6074\')\n    fig.savefig(out/\'stft_comparison.png\',dpi=170,bbox_inches=\'tight\');plt.close(fig)\n\n    # An audited, separately labelled band-power calculation on matched support.\n    bands=[(0,60),(60,250),(250,500),(500,1000),(1000,3000),(3000,8000),(8000,fs/2)]\n    labels=[\'0–60\',\'60–250\',\'250–500\',\'500–1k\',\'1k–3k\',\'3k–8k\',\'8k+\']\n    powers=[]\n    for y in ys:\n        f,p=welch(y,fs=fs,window=\'hann\',nperseg=4096,noverlap=2048,nfft=4096,detrend=\'constant\',scaling=\'density\')\n        fractions=[float(p[(f>=a)&(f<b)].sum()/p.sum()) for a,b in bands]\n        powers.append(fractions)\n    fig,ax=plt.subplots(figsize=(12,5.2),layout=\'constrained\')\n    x=np.arange(len(labels))\n    for i,(vals,label,color) in enumerate(zip(powers,[\'Engine reference\',\'Drum comparison\'],[TEAL,ORANGE])):\n        ax.bar(x+(i-.5)*.38,100*np.array(vals),width=.38,label=label,color=color)\n    ax.set_xticks(x,labels);ax.set(ylabel=\'Share of estimated signal power (%)\',xlabel=\'Frequency band (Hz)\',title=\'Band power on a common measurement grid\')\n    ax.legend(frameon=False);ax.grid(axis=\'y\',alpha=.16)\n    fig.text(.5,-.025,\'Welch PSD · matched duration and sample rate · separate from the original magnitude-based metric\',ha=\'center\',fontsize=12,color=\'#4c6074\')\n    fig.savefig(out/\'band_power_comparison.png\',dpi=170,bbox_inches=\'tight\');plt.close(fig)\n    return {\'common_sample_rate_hz\':fs,\'matched_duration_s\':duration,\'method\':\'Welch PSD; one 20 Hz high-pass; 4096 samples; 50% overlap; shared frequency grid\',\n            \'bands_hz\':bands,\'engine_power_shares\':powers[0],\'comparison_power_shares\':powers[1]}\n\ndef run(engine_path,comparison_path,out):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    figures=out.parent/\'figures\';figures.mkdir(parents=True,exist_ok=True)\n    engine=analyze_file(str(engine_path));comparison=analyze_file(str(comparison_path))\n    # Match the supplied log (drums first), then record the author\'s intended order.\n    log_shift,log_hints,log_severity=compare_results(comparison,engine)\n    intended_shift,intended_hints,intended_severity=compare_results(engine,comparison)\n    fixed=make_figures(engine,comparison,figures)\n    payload={\'audit_date\':\'2026-10-10\',\'project_date\':\'2026-03-23 (author-reported)\',\n        \'engine_recording\':summary(engine),\'drum_recording\':summary(comparison),\n        \'supplied_log_order\':{\'baseline\':\'drum\',\'test\':\'engine\',\'relative_shifts\':log_shift,\'heuristic_hints\':log_hints,\'heuristic_severity\':log_severity},\n        \'intended_order\':{\'baseline\':\'engine recording\',\'test\':\'drum loop\',\'relative_shifts\':intended_shift,\'heuristic_hints\':intended_hints,\'heuristic_severity\':intended_severity},\n        \'matched_support_power_audit\':fixed,\n        \'input_sha256\':{Path(engine_path).name:digest(engine_path),Path(comparison_path).name:digest(comparison_path)},\n        \'environment\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\'scipy\':scipy.__version__,\'matplotlib\':matplotlib.__version__,\'ffmpeg\':subprocess.check_output([\'ffmpeg\',\'-version\'],text=True).splitlines()[0]},\n        \'interpretation\':\'Unlabelled engine vs drum demonstration. Neither engine health nor fault-detection accuracy is established. Original scores are hand-tuned heuristics, not probabilities.\'}\n    (out/\'results.json\').write_text(json.dumps(payload,indent=2,ensure_ascii=False,allow_nan=False)+\'\\n\')\n    with (out/\'global_features.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.writer(f);w.writerow([\'metric\',\'engine_recording\',\'drum_recording\',\'definition\'])\n        for key in [\'duration_s\',\'sample_rate_hz\',\'dominant_freq\',\'rms\',\'crest_factor\',\'centroid\',\'flatness\',\'entropy\',\'harmonic_score\',\'peak_count\',\'anomaly_rate\',\'onset_density\']:\n            w.writerow([key,payload[\'engine_recording\'][key],payload[\'drum_recording\'][key],\'Original prototype definition\'])\n    for label,r in [(\'engine\',engine),(\'drum\',comparison)]:\n        with (out/f\'{label}_frame_features.csv\').open(\'w\',newline=\'\') as f:\n            w=csv.writer(f);keys=list(r[\'frames\']);w.writerow(keys)\n            w.writerows(zip(*(r[\'frames\'][key] for key in keys)))\n    import contextlib\n    with (out/\'prototype_reproduction.txt\').open(\'w\') as f,contextlib.redirect_stdout(f):\n        print(\'REPRODUCTION OF SUPPLIED ORDER: drum baseline, engine test.\\nNot validated engine fault inference.\')\n        print_original(\'BASELINE / REFERENCE (drum)\',comparison)\n        print_original(\'TEST / SUSPECT (engine)\',engine)\n        print(\'\\nRelative shifts:\',json.dumps(log_shift,indent=2))\n        print(\'Original heuristic severity:\',log_severity)\n        print(\'Original heuristic hints:\',log_hints)\n        print(\'\\nINTENDED ORDER: engine baseline, drum test\')\n        print(\'Relative shifts:\',json.dumps(intended_shift,indent=2))\n        print(\'Original heuristic severity:\',intended_severity)\n        print(\'Original heuristic hints:\',intended_hints)\n    print(json.dumps({\'engine_dominant_hz\':engine[\'global\'][\'dominant_freq\'],\'drum_dominant_hz\':comparison[\'global\'][\'dominant_freq\'],\n                     \'supplied_order_severity\':log_severity,\'matched_duration_s\':fixed[\'matched_duration_s\']},indent=2))\n    return payload\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--engine\',required=True,type=Path)\n    p.add_argument(\'--comparison\',required=True,type=Path)\n    p.add_argument(\'--output\',type=Path,default=ROOT/\'results\')\n    a=p.parse_args();run(a.engine,a.comparison,a.output)\n')


In [ ]:
from google.colab import files
uploaded=files.upload()
print('Uploaded filenames:')
for name in uploaded:
    print(' -',name)

# Set roles explicitly. Change these strings if your filenames differ.
ENGINE_REFERENCE='manshaofficial-diesel-car-engine-sound-111994.mp3'
COMPARISON_RECORDING='kamhunt-timbo-drumline-loop-103bpm-171091.mp3'
if ENGINE_REFERENCE==COMPARISON_RECORDING:
    raise ValueError('Choose two different recordings.')
for name in [ENGINE_REFERENCE,COMPARISON_RECORDING]:
    if name not in uploaded:
        raise ValueError('Set the filename variables to match your uploads: '+name)


In [ ]:
from IPython.display import display, Image
subprocess.run([sys.executable,str(work/'run_analysis.py'),
                '--engine',str(Path(ENGINE_REFERENCE).resolve()),
                '--comparison',str(Path(COMPARISON_RECORDING).resolve()),
                '--output',str(work/'results')],check=True)
for name in ['fft_comparison.png','stft_comparison.png','band_power_comparison.png']:
    display(Image(filename=str(work/'figures'/name)))
print((work/'results/prototype_reproduction.txt').read_text())
print('Evidence files:',work/'results')
